In [17]:
import pydicom
from pathlib import Path
from collections import defaultdict
import json

In [11]:
def count_four(folder):
    need = {"l-cc", "r-cc", "l-mlo", "r-mlo"}
    views = defaultdict(set)

    for f in Path(folder).rglob("DXm.*"):
        if not f.is_file():
            continue

        ds = pydicom.dcmread(str(f), stop_before_pixels=True, force=True)

        pid = ds.get("PatientID")
        lat = str(ds.get("ImageLaterality", "")).lower()

        v = str(ds.get("ViewPosition") or "").lower()
        if not v and getattr(ds, "ViewCodeSequence", None):
            v = str(ds.ViewCodeSequence[0].CodeMeaning).lower()

        if "cc" in v or "cranio" in v:
            v = "cc"
        elif "mlo" in v or "oblique" in v or "medio" in v:
            v = "mlo"
        else:
            v = ""

        if pid and lat in ("l", "r") and v:
            views[pid].add(f"{lat}-{v}")

    n_four = sum(need <= s for s in views.values())
    return n_four, len(views)

In [12]:
print(count_four("/raid/data01/deephealth/dh_dcm_ast"))

(163191, 186667)


In [13]:
print(count_four("/raid/data01/deephealth/dh_dh0new"))

(13998, 14166)


In [14]:
print(count_four("/raid/data01/deephealth/dh_dh2"))

(1967, 2367)


In [15]:
def count_four_json(folder):
    need = {"l-cc", "r-cc", "l-mlo", "r-mlo"}
    views = defaultdict(set)

    for f in Path(folder).glob("*.json"):
        with open(f) as fh:  # read-only
            m = json.load(fh)

        if m.get("label") == "Unknown":
            continue

        pid = m.get("PatientID")
        lat = str(m.get("ImageLaterality", "")).lower()
        v = str(m.get("View", "")).lower()

        if "cc" in v or "cranio" in v:
            v = "cc"
        elif "mlo" in v or "oblique" in v or "medio" in v:
            v = "mlo"
        else:
            v = ""

        if pid and lat in ("l", "r") and v:
            views[pid].add(f"{lat}-{v}")

    n_four = sum(need <= s for s in views.values())
    return n_four, len(views)




In [18]:
print(count_four_json("/raid/mpsych/OMAMA/DATA/data/2d/metadata"))

(82, 157800)
